In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.data.market_loader import MarketLoader
from src.data.synthetic_sofr_builder import build_term_sofr_curve

from src.term_structure.bootstrap_curve import (
    bootstrap_df_from_sofr,
    bootstrap_df_from_treasury_curve
)
from src.term_structure.curve_interpolation import (
    build_coupon_structure,
    zero_rate_to_df,
    interpolate_zero_curve
)

from src.term_structure.zero_curve_builder import (
    build_zero_curve_from_sofr_dfs,
    build_zero_curve_from_discount_curve
)

In [2]:
# downloading market curves
loader = MarketLoader()
curves = loader.loader_pipeline()

# building synthetic sofr curve from ON rates and futures
sofr_curve = build_term_sofr_curve(curves = curves)
sofr_curve

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..


,ON,3M,6M,1M
Date,,,,
2018-04-03,1.83,1.72,1.88,1.79
2018-04-04,1.74,1.68,1.86,1.72
2018-04-05,1.75,1.69,1.88,1.73
2018-04-06,1.75,1.70,1.86,1.73
2018-04-09,1.75,1.73,1.89,1.74
...,...,...,...,...
2026-04-15,3.72,3.62,3.60,3.69
2026-04-16,3.67,3.62,3.58,3.65
2026-04-17,3.65,3.61,3.56,3.64


In [3]:
# bootstrapping discount factors from sofr curve
df_bootstrap_from_sofr = bootstrap_df_from_sofr(
    sofr_curve = sofr_curve
)

df_bootstrap_from_sofr

,ON,3M,6M,1M
2018-04-03,0.999949,0.995718,0.990688,0.998511
2018-04-04,0.999952,0.995818,0.990786,0.998569
2018-04-05,0.999951,0.995793,0.990688,0.998560
2018-04-06,0.999951,0.995768,0.990786,0.998560
2018-04-09,0.999951,0.995694,0.990638,0.998552
...,...,...,...,...
2026-04-15,0.999897,0.991031,0.982318,0.996934
2026-04-16,0.999898,0.991031,0.982415,0.996968
2026-04-17,0.999899,0.991056,0.982511,0.996976
2026-04-20,0.999899,0.991031,0.982367,0.996984


In [4]:
# constructing zero curve from discount curve
zero_curve_mm = build_zero_curve_from_sofr_dfs(
    discount_curve = df_bootstrap_from_sofr
)

zero_curve_mm

,ON,3M,6M,1M
2018-04-03,1.829953,1.716313,1.871219,1.788666
2018-04-04,1.739958,1.676482,1.851404,1.718769
2018-04-05,1.749957,1.686440,1.871219,1.728754
2018-04-06,1.749957,1.696398,1.851404,1.728754
2018-04-09,1.749957,1.726270,1.881126,1.738740
...,...,...,...,...
2026-04-15,3.719808,3.603718,3.567984,3.684338
2026-04-16,3.669813,3.603718,3.548336,3.644460
2026-04-17,3.649815,3.593807,3.528687,3.634490
2026-04-20,3.629817,3.603718,3.558160,3.624521


In [5]:
# interpolate zero curve to coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

zero_curve_mm_interp = interpolate_zero_curve(
    zero_curve_df = zero_curve_mm,
    target_times = coupon_structure
)

### short-end of the curve
# df bootstrap using interpolated zero curve
df_mm_interp = zero_rate_to_df(
    zero_curve_interp = zero_curve_mm_interp
)

### long-end of the curve -> constructing discount factors from treasury par yields
# get treasury par yields
treasury_curve = curves['treasury']

# bootstrapping discount factors from treasury par yields merging money market dfs
df_bootstrap_full = bootstrap_df_from_treasury_curve(
    treasury_curve = treasury_curve,
    interpolated_dfs = df_mm_interp
)

# building zero curve from bootstrapped discount curve
zero_curve_full = build_zero_curve_from_discount_curve(
    discount_curve = df_bootstrap_full
)

zero_curve_full

,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0,...,25.5,26.0,26.5,27.0,27.5,28.0,28.5,29.0,29.5,30.0
2018-04-03,1.871219,2.080242,1.871219,2.272731,1.871219,1.871219,1.871219,1.871219,1.871219,2.624054,...,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,3.843541
2018-04-04,1.851404,2.060438,1.851404,2.273073,1.851404,1.851404,1.851404,1.851404,1.851404,2.635824,...,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,3.886078
2018-04-05,1.871219,2.060335,1.871219,2.293015,1.871219,1.871219,1.871219,1.871219,1.871219,2.666619,...,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,1.871219,3.962582
2018-04-06,1.851404,2.050484,1.851404,2.263046,1.851404,1.851404,1.851404,1.851404,1.851404,2.603962,...,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,1.851404,3.839084
2018-04-09,1.881126,2.070237,1.881126,2.282758,1.881126,1.881126,1.881126,1.881126,1.881126,2.623497,...,1.881126,1.881126,1.881126,1.881126,1.881126,1.881126,1.881126,1.881126,1.881126,3.834367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,3.567984,3.667100,3.567984,3.728633,3.567984,3.567984,3.567984,3.567984,3.567984,3.888006,...,3.567984,3.567984,3.567984,3.567984,3.567984,3.567984,3.567984,3.567984,3.567984,7.216122
2026-04-16,3.548336,3.657370,3.548336,3.749316,3.548336,3.548336,3.548336,3.548336,3.548336,3.900250,...,3.548336,3.548336,3.548336,3.548336,3.548336,3.548336,3.548336,3.548336,3.548336,7.489914
2026-04-17,3.528687,3.607988,3.528687,3.679413,3.528687,3.528687,3.528687,3.528687,3.528687,3.827106,...,3.528687,3.528687,3.528687,3.528687,3.528687,3.528687,3.528687,3.528687,3.528687,7.288699
2026-04-20,3.558160,3.617631,3.558160,3.688872,3.558160,3.558160,3.558160,3.558160,3.558160,3.846206,...,3.558160,3.558160,3.558160,3.558160,3.558160,3.558160,3.558160,3.558160,3.558160,7.198121
